# Next Day Wildfire Spread: Exploratory Data Analysis & Geospatial Inspection

## 1. Problem Formulation & Dataset Context
This notebook provides deep visual and statistical analysis of the **Google Research Next Day Wildfire Spread Dataset**.
- **Objective:** Given a  	imes 64	ext{ km}$ tile containing 12 remote sensing, meteorological, and elevation channels at Day $, predict the active wildfire burn front at Day +1$.
- **Tile Resolution:** 	ext{ km/pixel}$ ( 	imes 64 = 4,096$ pixels per tile).
- **Core ML Challenges:** Severe class imbalance ($<0.3\%$ active fire pixels), multi-band scale discrepancies, physical wind-topography coupling, and unobserved/masked sensor pixels ().

In [ ]:
import os
import sys
import glob
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Robust cross-platform path resolution
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "src").exists() else NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

try:
    from src.dataset import WildfireTFRecordDataset, FEATURE_NAMES, extract_example
    from src.utils import decompose_wind, load_json, DATA_DIR
except ImportError:
    from dataset import WildfireTFRecordDataset, FEATURE_NAMES, extract_example
    from utils import decompose_wind, load_json, DATA_DIR

print("Environment initialized successfully.")

## 2. Ingesting & Inspecting Sample Fire Events
We load sample records from the training split and search for an active fire incident where propagation occurs between Day $ and Day +1$.

In [ ]:
data_path = DATA_DIR / "raw" if (DATA_DIR / "raw").exists() else DATA_DIR
train_files = sorted(glob.glob(os.path.join(str(data_path), "*train_*.tfrecord")))
ds = WildfireTFRecordDataset(train_files, normalize=False)

# Find an interesting tile with active fire expansion
sample_inputs, sample_target, sample_mask = None, None, None
for inputs, target, mask in ds:
    prev_fire = inputs[FEATURE_NAMES.index("PrevFireMask")].sum().item()
    next_fire = target.sum().item()
    if prev_fire > 10 and next_fire > prev_fire:
        sample_inputs = inputs.numpy()
        sample_target = target.numpy()[0]
        sample_mask = mask.numpy()[0]
        print(f"Found active spread tile! PrevFire pixels: {int(prev_fire)}, NextFire pixels: {int(next_fire)}")
        break

## 3. Multi-Channel Geospatial Raster Visualization
We visualize all 12 input bands for this wildfire tile using specialized colormaps matching each physical variable.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

colormaps = {
    "elevation": "terrain",
    "pdsi": "BrBG",
    "NDVI": "YlGn",
    "pr": "Blues",
    "sph": "PuBu",
    "th": "hsv",
    "tmmn": "coolwarm",
    "tmmx": "plasma",
    "vs": "viridis",
    "erc": "OrRd",
    "population": "magma",
    "PrevFireMask": "hot"
}

for idx, name in enumerate(FEATURE_NAMES):
    ax = axes[idx]
    data = sample_inputs[idx]
    cmap = colormaps.get(name, "viridis")
    im = ax.imshow(data, cmap=cmap)
    ax.set_title(f"{name}", fontsize=11, fontweight="bold")
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle("Multi-Band Raster Features for Wildfire Tile (64x64 km)", fontsize=16, y=0.98)
plt.tight_layout()
plt.show()

## 4. Wildfire Spread Progression: Day $ vs. Day +1$
Visualizing how the active fire front progressed over 24 hours.

In [ ]:
prev_fire = sample_inputs[FEATURE_NAMES.index("PrevFireMask")]
next_fire = sample_target
new_spread = np.clip(next_fire - prev_fire, 0, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(prev_fire, cmap="hot")
axes[0].set_title(f"Day T: Previous Fire (Pixels: {int(prev_fire.sum())})", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(next_fire, cmap="hot")
axes[1].set_title(f"Day T+1: Ground Truth Target (Pixels: {int(next_fire.sum())})", fontweight="bold")
axes[1].axis("off")

axes[2].imshow(new_spread, cmap="Reds")
axes[2].set_title(f"New Burn Spread (Pixels: {int(new_spread.sum())})", fontweight="bold", color="crimson")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 5. Physics Analysis: Wind Direction and Propagation Dynamics
We decompose wind speed () and wind azimuth () into $ vector fields and overlay wind vectors on the elevation topography to examine physical spread alignment.

In [ ]:
elevation = sample_inputs[FEATURE_NAMES.index("elevation")]
wind_speed = sample_inputs[FEATURE_NAMES.index("vs")]
wind_dir = sample_inputs[FEATURE_NAMES.index("th")]

u, v = decompose_wind(wind_speed, wind_dir)

# Subsample grid for legible quiver arrows (every 4th pixel)
step = 4
Y, X = np.mgrid[0:64:step, 0:64:step]
U_sub = u[::step, ::step]
V_sub = v[::step, ::step]

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(elevation, cmap="terrain", alpha=0.8)
ax.contour(next_fire, levels=[0.5], colors="red", linewidths=2.5)
ax.quiver(X, Y, U_sub, V_sub, color="black", scale=40, width=0.005, alpha=0.7)
ax.set_title("Terrain Elevation with Wind Vectors and Day T+1 Fire Boundary (Red)", fontsize=12, fontweight="bold")
ax.axis("off")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Elevation (m)")
plt.tight_layout()
plt.show()

## 6. Training Set Scaling Distribution
Inspect the normalization parameters saved in  to ensure zero-leakage standard scaling.

In [ ]:
scalers = load_json("../configs/scaler_params.json")
print(f"{'Feature':<16} {'Mean':<12} {'Std':<12} {'Min':<12} {'Max':<12}")
print("-" * 64)
for name, s in scalers.items():
    print(f"{name:<16} {s['mean']:<12.3f} {s['std']:<12.3f} {s['min']:<12.3f} {s['max']:<12.3f}")